In [ ]:
import sys
from pathlib import Path
try:
    sys.path.insert(0, str(Path('__file__').resolve().parent))
except:
    pass
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent))

import preprocessing, shared
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

FIGS = shared.OUT / 'figures' / 'audit'
FIGS.mkdir(parents=True, exist_ok=True)


In [ ]:
fp = preprocessing.load_footprints()
fig, ax = plt.subplots(figsize=(12, 10))
fp.plot(ax=ax, color='#e8f4fb', edgecolor='#6baed6', linewidth=0.6)
ax.set_title('Raw Building Footprints'); ax.set_aspect('equal')
shared.save_fig(FIGS, '01_raw_footprints.png'); plt.show()


In [ ]:
df = preprocessing.load_excel_directions()
dir_counts = df['direction'].apply(lambda d: '|'.join(sorted(d)) if d else 'none').value_counts()
print(dir_counts.to_string())
fig, ax = plt.subplots(figsize=(8, 4))
dir_counts.plot.bar(ax=ax, color='steelblue')
ax.set_title('Direction Distribution'); ax.set_ylabel('Count')
plt.tight_layout(); shared.save_fig(FIGS, '02_direction_audit.png'); plt.show()


In [ ]:
dxf_labels = preprocessing.load_dxf_labels()
xs = [v[0] for v in dxf_labels.values()]
ys = [v[1] for v in dxf_labels.values()]
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(xs, ys, s=6, c='crimson', zorder=3)
for lbl, (x, y) in list(dxf_labels.items())[:50]:
    ax.text(x, y, lbl, fontsize=4, ha='center', va='bottom')
title = f'DXF Numeric Labels ({len(dxf_labels)} found)'
ax.set_title(title); ax.set_aspect('equal')
shared.save_fig(FIGS, '03_dxf_labels.png'); plt.show()


In [ ]:
H = preprocessing.estimate_rough_affine(dxf_labels, fp)
crosswalk = preprocessing.bipartite_label_match(dxf_labels, H, fp)
fp_attr, n_lab, n_dir, n_nodir, n_unlab = preprocessing.attribute_footprints(fp, crosswalk, df)
print(crosswalk.head(10).to_string(index=False))
print('Labelled:', n_lab, ' With dir:', n_dir, ' No dir:', n_nodir, ' Unlabelled:', n_unlab)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
fp_attr.plot(ax=ax, color='#eef3f7', edgecolor='#b0c4de', linewidth=0.5)
has_dir = fp_attr[fp_attr['direction'].apply(lambda d: isinstance(d,(list,tuple)) and len(d)>0)]
no_dir  = fp_attr[fp_attr['chapel_id'].notna() & fp_attr['direction'].apply(lambda d: len(d)==0 if isinstance(d,(list,tuple)) else True)]
has_dir.plot(ax=ax, color='#d4efdf', edgecolor='#27ae60', linewidth=0.8)
no_dir.plot(ax=ax, color='#fadbd8', edgecolor='#e74c3c', linewidth=0.8)
shared._draw_chapel_ids(ax, fp_attr, fontsize=4, color='#222', bbox=dict(boxstyle='round,pad=0.1',fc='white',alpha=0.7,lw=0))
ax.set_title('Attributed Footprints (green=has dir, red=no dir)'); ax.set_aspect('equal')
shared.save_fig(FIGS, '04_attributed_footprints.png'); plt.show()


In [ ]:
doors_gdf, doors_pts = preprocessing.place_doors(fp_attr)
fig, ax = plt.subplots(figsize=(14, 12))
fp_attr.plot(ax=ax, color='#eef3f7', edgecolor='#b0c4de', linewidth=0.5)
for _, row in doors_gdf.iterrows():
    xs2, ys2 = row.geometry.xy
    ax.plot(xs2, ys2, color=shared.DIR_CLR.get(row['direction'],'#aaa'), linewidth=1.8, zorder=4)
ax.legend(handles=shared.direction_legend_patches(), fontsize=8, loc='lower left')
ax.set_aspect('equal'); ax.set_title(f'Door Placement ({len(doors_gdf)} doors)')
shared.save_fig(FIGS, '05_doors.png'); plt.show()


In [ ]:
sample = fp_attr[fp_attr['chapel_id'].notna()].head(30)
xmin, ymin, xmax, ymax = sample.total_bounds
fig, ax = plt.subplots(figsize=(12, 10))
fp_attr.plot(ax=ax, color='#eef3f7', edgecolor='#b0c4de', linewidth=0.5)
for _, row in doors_gdf.iterrows():
    xs2, ys2 = row.geometry.xy
    ax.plot(xs2, ys2, color=shared.DIR_CLR.get(row['direction'],'#aaa'), linewidth=2.0, zorder=4)
shared._draw_chapel_ids(ax, fp_attr, fontsize=5, color='#222', bbox=dict(boxstyle='round,pad=0.1',fc='white',alpha=0.7,lw=0))
ax.set_xlim(xmin-20, xmax+20); ax.set_ylim(ymin-20, ymax+20)
ax.set_title('Zoomed — Doors on Labelled Buildings'); ax.set_aspect('equal')
shared.save_fig(FIGS, '06_doors_zoom.png'); plt.show()


In [ ]:
dem = shared.load_dem()
hs  = shared.hillshade(dem['disp'])
fig, ax = plt.subplots(figsize=(14, 12))
ax.imshow(dem['disp'], extent=dem['extent'], origin='upper', cmap='terrain', alpha=0.65,
          vmin=dem['e_min'], vmax=dem['e_max'], rasterized=True)
ax.imshow(hs, extent=dem['extent'], origin='upper', cmap='gray', alpha=0.3, rasterized=True)
ax.set_title('DEM Hillshade'); shared.save_fig(FIGS, '07_dem.png'); plt.show()


In [ ]:
doors_pts = shared.sample_dem_at_doors(doors_pts)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(doors_pts['elevation_m'].dropna(), bins=30, color='steelblue', edgecolor='white')
ax.set_title('Door Elevation Distribution (m)'); ax.set_xlabel('Elevation (m)'); ax.set_ylabel('Count')
plt.tight_layout(); shared.save_fig(FIGS, '08_door_elevation.png'); plt.show()
